In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/Capstone_IoT'
RAW = os.path.join(BASE, 'Data/Raw')

print("Files in Data/Raw/:")
for f in sorted(os.listdir(RAW)):
    path = os.path.join(RAW, f)
    size_mb = os.path.getsize(path) / (1024**2)
    print(f"  {f}  ({size_mb:.1f} MB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in Data/Raw/:
  Merged01.csv  (140.3 MB)
  Merged32.csv  (138.7 MB)
  Merged63.csv  (84.4 MB)


In [5]:
import pandas as pd

sample_file = os.path.join(RAW, 'Merged01.csv')

# Read just the first 5 rows to see columns without loading everything
peek = pd.read_csv(sample_file, nrows=5)
print(f"Shape of peek: {peek.shape}")
print(f"\nColumns ({len(peek.columns)}):")
for i, col in enumerate(peek.columns):
    print(f"  {i:3d}  {col}")

Shape of peek: (5, 40)

Columns (40):
    0  Header_Length
    1  Protocol Type
    2  Time_To_Live
    3  Rate
    4  fin_flag_number
    5  syn_flag_number
    6  rst_flag_number
    7  psh_flag_number
    8  ack_flag_number
    9  ece_flag_number
   10  cwr_flag_number
   11  ack_count
   12  syn_count
   13  fin_count
   14  rst_count
   15  HTTP
   16  HTTPS
   17  DNS
   18  Telnet
   19  SMTP
   20  SSH
   21  IRC
   22  TCP
   23  UDP
   24  DHCP
   25  ARP
   26  ICMP
   27  IGMP
   28  IPv
   29  LLC
   30  Tot sum
   31  Min
   32  Max
   33  AVG
   34  Std
   35  Tot size
   36  IAT
   37  Number
   38  Variance
   39  Label


In [6]:
print("Loading Merged01.csv fully...")
df1 = pd.read_csv(sample_file)
print(f"Shape: {df1.shape}")
print(f"Memory usage: {df1.memory_usage(deep=True).sum() / (1024**2):.1f} MB")
print(f"\nDtypes summary:")
print(df1.dtypes.value_counts())

Loading Merged01.csv fully...
Shape: (712311, 40)
Memory usage: 255.6 MB

Dtypes summary:
float64    30
int64       9
object      1
Name: count, dtype: int64


In [7]:
# The README says the label column is 'label' or 'Label' — find it
label_candidates = [c for c in df1.columns if c.lower() == 'label']
print(f"Label column candidates: {label_candidates}")

if label_candidates:
    LABEL_COL = label_candidates[0]
    print(f"\nUsing label column: '{LABEL_COL}'")
    print(f"\nClass distribution in Merged01.csv:")
    print(df1[LABEL_COL].value_counts())
    print(f"\nTotal unique classes: {df1[LABEL_COL].nunique()}")
else:
    print("WARNING: no label column found — printing all columns for inspection")
    print(df1.columns.tolist())

Label column candidates: ['Label']

Using label column: 'Label'

Class distribution in Merged01.csv:
Label
DDOS-ICMP_FLOOD            108662
DDOS-UDP_FLOOD              82011
DDOS-TCP_FLOOD              68289
DDOS-PSHACK_FLOOD           62171
DDOS-RSTFINFLOOD            61652
DDOS-SYN_FLOOD              61460
DDOS-SYNONYMOUSIP_FLOOD     54749
DOS-UDP_FLOOD               50371
DOS-TCP_FLOOD               40391
DOS-SYN_FLOOD               30620
BENIGN                      16577
MIRAI-GREETH_FLOOD          15135
MIRAI-UDPPLAIN              13342
MIRAI-GREIP_FLOOD           11187
DDOS-ICMP_FRAGMENTATION      6784
VULNERABILITYSCAN            5805
MITM-ARPSPOOFING             4590
DDOS-ACK_FRAGMENTATION       4308
DDOS-UDP_FRAGMENTATION       4264
DNS_SPOOFING                 2738
RECON-HOSTDISCOVERY          2045
RECON-OSSCAN                 1433
RECON-PORTSCAN               1251
DOS-HTTP_FLOOD               1113
DDOS-HTTP_FLOOD               416
DDOS-SLOWLORIS                354
DICTIONAR

In [8]:
print(f"Total NaN count across all columns: {df1.isna().sum().sum()}")
print(f"\nColumns with NaNs:")
nan_cols = df1.isna().sum()
nan_cols = nan_cols[nan_cols > 0]
if len(nan_cols) > 0:
    print(nan_cols)
else:
    print("  None")

print(f"\nFirst 3 rows (all columns):")
pd.set_option('display.max_columns', None)
print(df1.head(3))

Total NaN count across all columns: 22

Columns with NaNs:
Std         11
Variance    11
dtype: int64

First 3 rows (all columns):
   Header_Length  Protocol Type  Time_To_Live          Rate  fin_flag_number  \
0          19.92              6         63.36  25893.962218              0.0   
1           0.00             47         64.00   3703.841331              0.0   
2           7.92             17         65.91  19673.095685              0.0   

   syn_flag_number  rst_flag_number  psh_flag_number  ack_flag_number  \
0              0.0              0.0             0.99             0.99   
1              0.0              0.0             0.00             0.00   
2              0.0              0.0             0.00             0.00   

   ece_flag_number  cwr_flag_number  ack_count  syn_count  fin_count  \
0              0.0              0.0         99          0          0   
1              0.0              0.0          0          0          0   
2              0.0              0.0    

In [9]:
import pandas as pd
import os

print("Loading all 3 Merged files...")
all_files = sorted([f for f in os.listdir(RAW) if f.endswith('.csv')])
print(f"Files to load: {all_files}")

dfs = []
for f in all_files:
    path = os.path.join(RAW, f)
    print(f"  Loading {f}...")
    dfs.append(pd.read_csv(path))

df = pd.concat(dfs, ignore_index=True)
del dfs  # free memory
print(f"\nCombined shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

Loading all 3 Merged files...
Files to load: ['Merged01.csv', 'Merged32.csv', 'Merged63.csv']
  Loading Merged01.csv...
  Loading Merged32.csv...
  Loading Merged63.csv...

Combined shape: (1844539, 40)
Memory: 661.9 MB


In [10]:
LABEL_COL = 'Label'

print(f"Combined class distribution ({df[LABEL_COL].nunique()} classes):")
print(df[LABEL_COL].value_counts())

print(f"\nTotal NaN count: {df.isna().sum().sum()}")
nan_cols = df.isna().sum()
nan_cols = nan_cols[nan_cols > 0]
if len(nan_cols) > 0:
    print(f"NaN columns:\n{nan_cols}")


Combined class distribution (34 classes):
Label
DDOS-ICMP_FLOOD            281580
DDOS-UDP_FLOOD             211566
DDOS-TCP_FLOOD             176359
DDOS-PSHACK_FLOOD          161033
DDOS-RSTFINFLOOD           159330
DDOS-SYN_FLOOD             159079
DDOS-SYNONYMOUSIP_FLOOD    141097
DOS-UDP_FLOOD              130715
DOS-TCP_FLOOD              105150
DOS-SYN_FLOOD               79624
BENIGN                      43265
MIRAI-GREETH_FLOOD          38997
MIRAI-UDPPLAIN              34740
MIRAI-GREIP_FLOOD           29432
DDOS-ICMP_FRAGMENTATION     17695
VULNERABILITYSCAN           14720
MITM-ARPSPOOFING            11875
DDOS-UDP_FRAGMENTATION      11332
DDOS-ACK_FRAGMENTATION      11231
DNS_SPOOFING                 7058
RECON-HOSTDISCOVERY          5227
RECON-OSSCAN                 3758
RECON-PORTSCAN               3182
DOS-HTTP_FLOOD               2814
DDOS-HTTP_FLOOD              1106
DDOS-SLOWLORIS                927
DICTIONARYBRUTEFORCE          513
BROWSERHIJACKING              248
